In [2]:
# Colab Setup for I3D action recognition
!pip install -q tensorflow tensorflow_hub opencv-python imageio

import tensorflow as tf
import tensorflow_hub as hub
import cv2, numpy as np
import imageio
from IPython.display import HTML, display

# Load pre-trained I3D model
i3d = hub.load("https://tfhub.dev/deepmind/i3d-kinetics-400/1").signatures['default']

# Load label map from Kinetics-400 classes
LABELS_URL = "https://storage.googleapis.com/deepmind-media/DNN-models/kinetics-classnames.txt"
labels = tf.keras.utils.get_file('labels.txt', LABELS_URL)
with open(labels) as f:
    kinetics_labels = [line.strip() for line in f.readlines()]

def predict_video(video_path):
    vid = imageio.get_reader(video_path, 'ffmpeg')
    frames = []
    for i, frame in enumerate(vid):
        frame = cv2.resize(frame, (224, 224))
        frames.append(frame)
        if len(frames) == 64:
            inp = np.expand_dims(np.array(frames), 0) / 255.0
            outputs = i3d(tf.constant(inp))['default']
            topk = outputs.numpy().argsort()[0][-5:][::-1]
            display(HTML(f"<b>Top predictions for chunk {i//64}:<br>" +
                         "<br>".join([f"{kinetics_labels[k]} — {outputs[0][k]:.3f}" for k in topk]) +
                         "</b><br><br>"))
            frames = []
    vid.close()

print("### Analyze fight.mp4 ###")
predict_video('fight.mp4')

print("### Analyze nonfight.mp4 ###")
predict_video('nonfight.mp4')


Exception: URL fetch failure on https://storage.googleapis.com/deepmind-media/DNN-models/kinetics-classnames.txt: 404 -- Not Found

In [1]:
# Colab Setup for PyTorch Fight-Detection
!pip install fight-detection pytube
!pip install torch torchvision

from fight_detection import Fight_utils

# Process fight.mp4
print("Processing fight.mp4...")
Fight_utils.fightDetection(inputPath='fight.mp4', seq=16, skip=0,
                           outputPath='output_fight.mp4', showInfo=True)

# Process nonfight.mp4
print("Processing nonfight.mp4...")
Fight_utils.fightDetection(inputPath='nonfight.mp4', seq=16, skip=0,
                           outputPath='output_nonfight.mp4', showInfo=True)

# Display outputs
from IPython.display import HTML, Video
print("Output fight.mp4:")
display(Video('output_fight.mp4', embed=True))
print("Output nonfight.mp4:")
display(Video('output_nonfight.mp4', embed=True))


  if event.key is 'enter':



FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1MWDeLnpEaZDrKK-OjmzvYLxfjwp-GDcp

but Gdown can't. Please check connections and permissions.

In [2]:
# Colab Setup
!pip install -q tensorflow==2.0.0 opencv-python numpy scikit-image pillow

import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from IPython.display import Video

# Define basic CNN-LSTM model (simplified)
def build_model(input_shape=(16, 224, 224, 3)):
    model = models.Sequential([
        layers.TimeDistributed(layers.Conv2D(32, (3,3), activation='relu'), input_shape=input_shape),
        layers.TimeDistributed(layers.MaxPooling2D((2,2))),
        layers.TimeDistributed(layers.Conv2D(64, (3,3), activation='relu')),
        layers.TimeDistributed(layers.MaxPooling2D((2,2))),
        layers.TimeDistributed(layers.Flatten()),
        layers.LSTM(128),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')  # output: violence probability
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_model()

# Placeholder: load pretrained weights if available; else, this will infer randomly
# model.load_weights('path_to_pretrained_weights.h5')

def preprocess_video(path, seq_len=16):
    cap = cv2.VideoCapture(path)
    frames = []
    sequences = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (224,224))
        frames.append(frame / 255.0)
        if len(frames) == seq_len:
            sequences.append(np.array(frames))
            frames = []
    cap.release()
    return np.array(sequences)

def detect(path, label):
    seqs = preprocess_video(path)
    preds = model.predict(seqs, verbose=0)
    print(f"{label} – average violence probability: {preds.mean():.3f}")

# Run detection
detect('fight.mp4', 'fight.mp4')
detect('nonfight.mp4', 'nonfight.mp4')


ERROR: Could not find a version that satisfies the requirement tensorflow==2.0.0 (from versions: 2.12.0rc0, 2.12.0rc1, 2.12.0, 2.12.1, 2.13.0rc0, 2.13.0rc1, 2.13.0rc2, 2.13.0, 2.13.1, 2.14.0rc0, 2.14.0rc1, 2.14.0, 2.14.1, 2.15.0rc0, 2.15.0rc1, 2.15.0, 2.15.0.post1, 2.15.1, 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==2.0.0


  super().__init__(**kwargs)



KeyboardInterrupt: 

In [3]:
# Colab Setup
!pip install -q tensorflow opencv-python imageio

import tensorflow as tf
import cv2, numpy as np
from IPython.display import Video

# Define a basic 3D-CNN (simplified placeholder)
def build_c3d(seq_len=16, height=112, width=112, channels=3):
    model = tf.keras.Sequential([
        layers.Conv3D(32, (3,3,3), activation='relu', input_shape=(seq_len, height, width, channels)),
        layers.MaxPooling3D((1,2,2)),
        layers.Conv3D(64, (3,3,3), activation='relu'),
        layers.MaxPooling3D((2,2,2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_c3d()

# Function to load 16-frame chunks
def preprocess_c3d(path, seq_len=16):
    vid = cv2.VideoCapture(path)
    frames = []
    seqs = []
    while True:
        ret, frame = vid.read()
        if not ret: break
        frame = cv2.resize(frame, (112,112))
        frames.append(frame / 255.0)
        if len(frames) == seq_len:
            seqs.append(np.array(frames))
            frames = []
    vid.release()
    return np.array(seqs)

def detect_c3d(path, label):
    seqs = preprocess_c3d(path)
    preds = model.predict(seqs, verbose=0)
    print(f"{label} – avg violence prob: {preds.mean():.3f}")

detect_c3d('fight.mp4', 'fight.mp4')
detect_c3d('nonfight.mp4', 'nonfight.mp4')


  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



fight.mp4 – avg violence prob: 0.482
nonfight.mp4 – avg violence prob: 0.473


In [9]:
# Install necessary packages
!pip install -q torch torchvision
!pip install -q git+https://github.com/facebookresearch/pytorchvideo.git

import torch
try:
    # Import necessary modules from pytorchvideo and torchvision
    from pytorchvideo.models.movinet import create_movinet
    from torchvision.transforms import (
        Compose,
        Lambda,
        Resize,
        Normalize
    )
    from torchvision.transforms._transforms_video import (
        ToTensorVideo,
    )
except ImportError:
    print("Failed to import necessary modules. Please ensure torch, torchvision, and pytorchvideo are installed correctly.")
    create_movinet = None
    Compose = None # Set to None if import fails to avoid NameError
    Lambda = None
    Resize = None
    Normalize = None
    ToTensorVideo = None


import cv2
import numpy as np

# Load pretrained model (fine-tuned weights to be loaded manually if available)
# Using MoViNet A3 as an example, adjust if you have a specific pre-trained model
model = None # Initialize model to None
if create_movinet is not None:
    try:
        model = create_movinet("movinet_a3", pretrained=True)
        # Adjust head for binary classification (fight vs non-fight) if needed
        # For a pretrained model, you might need to replace the final layer
        # num_ftrs = model.classifier[0].in_features
        # model.classifier[0] = torch.nn.Linear(num_ftrs, 1)
    except Exception as e:
        print(f"Error loading MoViNet model: {e}")
        model = None
else:
    print("MoViNet model creation function not found.")


# Check for CUDA and move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if model is not None:
    model.to(device)
    model.eval()
else:
    print("Model not loaded, skipping prediction.")


# Frame preprocessing - Adjusted for MoViNet
# MoViNet typically uses a stream buffer and processes frames sequentially
# The preprocessing needs to prepare individual frames or small clips for the model
# This is a simplified example and might need adjustments based on the specific MoViNet implementation
transform = None # Initialize transform to None
if Compose is not None and ToTensorVideo is not None and Lambda is not None and Resize is not None and Normalize is not None:
    transform = Compose([
        ToTensorVideo(), # Converts to C, T, H, W
        Lambda(lambda x: x / 255.0), # Normalize to [0, 1]
        Resize((224, 224)),
        Normalize([0.45, 0.45, 0.45], [0.225, 0.225, 0.225]),
        Lambda(lambda x: x.permute(1, 0, 2, 3)) # Permute to T, C, H, W
    ])
else:
    print("Failed to define transformations due to missing imports.")


def predict_violence(video_path):
    if model is None or transform is None:
        print("Model or transformations not available for prediction.")
        return 0.0

    cap = cv2.VideoCapture(video_path)
    scores = []
    # Initialize state for streaming model
    state = None
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame is not None:
                # Convert BGR to RGB and apply transformations
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                # Transform expects a tensor, so convert frame to tensor, then apply transform
                # MoViNet expects (N, C, T, H, W), with T=1 for streaming
                frame_tensor = torch.from_numpy(frame_rgb).permute(2, 0, 1).unsqueeze(1).float() # C, T=1, H, W
                frame_tensor = transform(frame_tensor) # Apply the defined transformations
                frame_tensor = frame_tensor.unsqueeze(0).to(device) # Add batch dimension N=1

                # Forward pass through the model with state
                if state is None:
                    output, state = model(frame_tensor)
                else:
                    output, state = model(frame_tensor, state)

                # Assuming the model outputs a single score for violence probability
                # The output shape might vary based on the specific MoViNet head
                # You might need to adjust this part to get the correct score
                score = output.squeeze().item()
                scores.append(score)

    cap.release()
    return np.mean(scores) if scores else 0.0

print("fight.mp4 avg score:", predict_violence("fight.mp4"))
print("nonfight.mp4 avg score:", predict_violence("nonfight.mp4"))

  Preparing metadata (setup.py) ... done
Failed to import necessary modules. Please ensure torch, torchvision, and pytorchvideo are installed correctly.
MoViNet model creation function not found.
Model not loaded, skipping prediction.
Failed to define transformations due to missing imports.
Model or transformations not available for prediction.
fight.mp4 avg score: 0.0
Model or transformations not available for prediction.
nonfight.mp4 avg score: 0.0


In [10]:
!pip install -q tensorflow opencv-python imageio

import tensorflow as tf
import cv2
import numpy as np

# Load the pretrained C3D model (ensure you provide the correct path or loading mechanics)
model = tf.keras.models.load_model('path_to_pretrained_c3d.h5')

def detect_violence_c3d(video_path, seq_len=16):
    cap = cv2.VideoCapture(video_path)
    frames, scores = [], []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (112, 112))
        frames.append(frame / 255.0)
        if len(frames) == seq_len:
            clip = np.expand_dims(np.array(frames), axis=0)
            score = model.predict(clip)[0][0]
            scores.append(score)
            frames = []
    cap.release()
    return np.mean(scores) if scores else 0.0

print("fight.mp4 avg score:", detect_violence_c3d("fight.mp4"))
print("nonfight.mp4 avg score:", detect_violence_c3d("nonfight.mp4"))


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'path_to_pretrained_c3d.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [1]:
# --- Colab Setup ---
!pip -q install ultralytics opencv-python

# Clone the repo that already contains trained weights (.pt files)
!git clone -q https://github.com/Musawer1214/Fight-Violence-detection-yolov8.git
%cd Fight-Violence-detection-yolov8

# Choose which weights to use (small = a bit heavier but usually better; nano = fastest)
WEIGHTS = "yolo_small_weights.pt"   # or "Yolo_nano_weights.pt"

# Upload your videos (expects fight.mp4 and nonfight.mp4)
from google.colab import files
print("Upload fight.mp4 and nonfight.mp4…")
_ = files.upload()

from ultralytics import YOLO
import os, glob
from IPython.display import Video, display

model = YOLO(WEIGHTS)  # loads the trained YOLOv8 model

def run_and_summarize(src_path):
    # Run prediction; Ultralytics auto-saves annotated video under runs/detect/...
    results = model.predict(
        source=src_path,
        imgsz=640,
        conf=0.25,   # raise to reduce false positives
        iou=0.5,
        save=True,
        stream=False,
        vid_stride=1,
        max_det=100
    )

    # Count total detections across all frames
    total, frames = 0, 0
    for r in results:
        frames += 1
        if hasattr(r, "boxes") and r.boxes is not None:
            total += len(r.boxes)

    # Find the saved annotated video
    out_dir = results[0].save_dir if isinstance(results, list) and results else None
    out_vids = sorted(glob.glob(os.path.join(str(out_dir), "*.mp4"))) if out_dir else []
    out_path = out_vids[0] if out_vids else None

    verdict = "FIGHT DETECTED ✅" if total > 0 else "NO FIGHT ❌"
    print(f"{os.path.basename(src_path)} → detections: {total} over {frames} frames → {verdict}")
    if out_path:
        display(Video(out_path, embed=True))
    return total, frames, out_path

print("\n--- Analyzing fight.mp4 ---")
run_and_summarize("fight.mp4")

print("\n--- Analyzing nonfight.mp4 ---")
run_and_summarize("nonfight.mp4")


fatal: destination path 'Fight-Violence-detection-yolov8' already exists and is not an empty directory.
/content/Fight-Violence-detection-yolov8
Upload fight.mp4 and nonfight.mp4…


Saving fight.mp4 to fight (2).mp4
Saving nonfight.mp4 to nonfight (1).mp4

--- Analyzing fight.mp4 ---

WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/991) /content/Fight-Violence-detection-yolov8/fight.mp4: 640x384 1 violence, 506.1ms
video 1/1 (frame 2/991) /content/Fight-Violence-detection-yolov8/fight.mp4: 640x384 1 violence, 408.8ms
video 1/1 (frame 3/991) /content/Fight-Violence-detection-yolov8/fight.mp4: 640x384 1 violence, 368.2ms
video 1/1 (frame 4/991) /content/Fight-Vi

(405, 374, None)

In [2]:
# --- Colab Setup ---
!pip install -q tensorflow opencv-python numpy
!git clone -q https://github.com/abduulrahmankhalid/Real-Time-Violence-Detection.git
%cd Real-Time-Violence-Detection

# If there are pretrained weights, they should be in repo; else, you'll train or hack
# For this demo, we'll infer (random), but code will run end-to-end.

import cv2, numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, TimeDistributed, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from IPython.display import Video, display
from google.colab import files
print("Upload fight.mp4 and nonfight.mp4 …")
_ = files.upload()

# Build model structure (as per repo)
frame_shape = (224, 224, 3)
seq_len = 16

base = MobileNetV2(include_top=False, input_shape=frame_shape, pooling='avg')
inp = Input((seq_len,) + frame_shape)
x = TimeDistributed(base)(inp)
x = Bidirectional(LSTM(64))(x)
out = Dense(1, activation='sigmoid')(x)
model = Model(inp, out)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# If pretrained weights exist:
# model.load_weights('pretrained_weights.h5')

def preprocess_video(path):
    cap = cv2.VideoCapture(path)
    seqs = []
    buf = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.resize(frame, frame_shape[:2]) / 255.0
        buf.append(frame)
        if len(buf) == seq_len:
            seqs.append(buf)
            buf = []
    cap.release()
    return np.array(seqs)

def test(path, label):
    seqs = preprocess_video(path)
    if len(seqs) == 0:
        print(f"No full sequence in {path}")
        return
    preds = model.predict(seqs, verbose=0)
    avg = preds.mean()
    verdict = "FIGHT" if avg >= 0.5 else "NON-FIGHT"
    print(f"{label}: avg score {avg:.3f} → {verdict}")

test("fight.mp4", "fight.mp4")
test("nonfight.mp4", "nonfight.mp4")


/content/Fight-Violence-detection-yolov8/Real-Time-Violence-Detection
Upload fight.mp4 and nonfight.mp4 …


Saving fight.mp4 to fight.mp4
Saving nonfight.mp4 to nonfight.mp4
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
fight.mp4: avg score 0.610 → FIGHT
nonfight.mp4: avg score 0.621 → FIGHT


In [ ]:
# ========= Colab setup =========
!pip -q install "torch>=2.2" "torchvision>=0.17" --extra-index-url https://download.pytorch.org/whl/cu121
!pip -q install opencv-python imageio

import os, cv2, math, numpy as np, torch
from torchvision.models.video import r3d_18, R3D_18_Weights   # CHANGE HERE to try mc3_18 or r2plus1d_18
from IPython.display import Video, display
import imageio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ====== Tunables ======
CLIP_LEN = 16           # model expects 16 frames
STRIDE = 8              # slide window (frames)
PEAK_THR = 0.55         # any clip peak fight prob above this helps trigger
MEAN_THR = 0.35         # mean fight prob across clips
FRAC_THR = 0.25         # fraction of clips considered "violent"
FONT = cv2.FONT_HERSHEY_SIMPLEX

# ====== Load model & preprocessing / labels ======
weights = R3D_18_Weights.DEFAULT
model = r3d_18(weights=weights).eval().to(device)
preprocess = weights.transforms()
KINETICS_CLASSES = weights.meta["categories"]

# Build violence class filter
VIOLENT_KEYWORDS = [
    "punch", "kick", "wrestl", "slap", "fight", "boxing",
    "karate", "taekwondo", "judo", "kung fu", "mma", "mixed martial arts",
    "kickboxing", "capoeira"
]
EXCLUDE_PHRASES = ["punching bag", "arm wrestling"]  # non-violent lookalikes

def is_violent_label(lbl: str):
    l = lbl.lower()
    if any(ex in l for ex in EXCLUDE_PHRASES): return False
    return any(k in l for k in VIOLENT_KEYWORDS)

VIOLENT_INDICES = [i for i, c in enumerate(KINETICS_CLASSES) if is_violent_label(c)]
print(f"Violence-class count: {len(VIOLENT_INDICES)}")

def read_video_cv2(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)   # RGB for TorchVision
        frames.append(f)
    cap.release()
    return np.array(frames), fps

def make_clips(frames_rgb, clip_len=16, stride=8):
    T = len(frames_rgb)
    if T == 0:
        return []
    clips = []
    for start in range(0, max(1, T - clip_len + 1), stride):
        end = min(T, start + clip_len)
        clip = frames_rgb[start:end]
        if len(clip) < clip_len:
            # pad last frame
            pad = np.repeat(clip[-1][None, ...], clip_len - len(clip), axis=0)
            clip = np.concatenate([clip, pad], axis=0)
        clips.append((start, end, clip))
    return clips

def clip_to_tensor_TCHW(clip_np):
    # clip_np: (T, H, W, C) uint8
    # TorchVision transforms expect (T, C, H, W) uint8 or float in [0,1]
    t = torch.from_numpy(clip_np).permute(0,3,1,2)  # TCHW
    return t

@torch.no_grad()
def score_video(path):
    frames_rgb, fps = read_video_cv2(path)
    clips = make_clips(frames_rgb, CLIP_LEN, STRIDE)
    violent_probs, top_preds = [], []

    for (s, e, clip) in clips:
        vid_TCHW = clip_to_tensor_TCHW(clip)                      # (T,C,H,W), uint8
        inp = preprocess(vid_TCHW).unsqueeze(0).to(device)         # -> (1,C,T,H,W)
        logits = model(inp)[0]                                     # (400,)
        probs = torch.softmax(logits, dim=0).cpu().numpy()

        # Sum probabilities of all violent categories
        vprob = float(np.sum(probs[VIOLENT_INDICES]))
        violent_probs.append(vprob)

        # For logging
        topk = probs.argsort()[-3:][::-1]
        top_preds.append([(KINETICS_CLASSES[i], float(probs[i])) for i in topk])

    violent_probs = np.array(violent_probs, dtype=np.float32)
    peak, mean = violent_probs.max() if len(violent_probs) else 0.0, violent_probs.mean() if len(violent_probs) else 0.0
    frac = float((violent_probs > 0.5).mean()) if len(violent_probs) else 0.0
    fight = (peak >= PEAK_THR) or ((mean >= MEAN_THR) and (frac >= FRAC_THR))
    return fight, dict(peak=peak, mean=mean, frac=frac, per_clip=violent_probs.tolist(), top=top_preds, fps=fps, nclips=len(violent_probs))

def annotate_and_save(src_path, verdict, stats, out_path):
    cap = cv2.VideoCapture(src_path)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    out = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
    banner = f"{'FIGHT' if verdict else 'NO FIGHT'} | peak:{stats['peak']:.2f} mean:{stats['mean']:.2f} frac:{stats['frac']:.2f}"
    color = (0, 200, 0) if verdict else (0, 0, 255)  # BGR for cv2 text

    while True:
        ok, frame = cap.read()
        if not ok: break
        cv2.rectangle(frame, (8, 8), (w - 8, 80), (0, 0, 0), -1)
        cv2.putText(frame, banner, (20, 55), FONT, 1.2, color, 3, cv2.LINE_AA)
        out.write(frame)
    cap.release()
    out.release()

def run_one(path):
    verdict, stats = score_video(path)
    out_path = os.path.splitext(os.path.basename(path))[0] + "_r3d18_annotated.mp4"
    annotate_and_save(path, verdict, stats, out_path)
    print(os.path.basename(path), "→",
          f"clips:{stats['nclips']}  peak:{stats['peak']:.3f}  mean:{stats['mean']:.3f}  frac>0.5:{stats['frac']:.2f}",
          "→", "FIGHT ✅" if verdict else "NO FIGHT ❌")
    if stats['nclips']:
        first = stats['top'][0]
        print("Top-3 (first clip):", ", ".join([f"{l}:{p:.2f}" for l,p in first]))
    display(Video(out_path, embed=True))

# ====== Upload your 2 videos ======
from google.colab import files
print("Upload fight.mp4 and nonfight.mp4 …")
_ = files.upload()

# ====== Run ======
run_one("fight.mp4")
run_one("nonfight.mp4")


Device: cpu
Violence-class count: 13
Upload fight.mp4 and nonfight.mp4 …


Saving fight.mp4 to fight (1).mp4
Saving nonfight.mp4 to nonfight (1).mp4


In [1]:
# ========= Colab setup =========
!pip -q install "tensorflow>=2.13,<2.18" tensorflow-hub opencv-python imageio

import os, cv2, numpy as np, tensorflow as tf, tensorflow_hub as hub, imageio
from urllib.request import urlopen
from IPython.display import Video, display

# ====== Tunables ======
CLIP_LEN = 64           # I3D typically uses 64 frames
STRIDE = 32             # slide window
PEAK_THR = 0.55
MEAN_THR = 0.35
FRAC_THR = 0.25
FONT = cv2.FONT_HERSHEY_SIMPLEX

# ====== Load model & labels ======
i3d = hub.load("https://tfhub.dev/deepmind/i3d-kinetics-400/1").signatures['default']
# Official Kinetics-400 label map used by the TF-Hub tutorial:
labels_url = "https://raw.githubusercontent.com/deepmind/kinetics-i3d/master/data/label_map.txt"
labels = [l.decode("utf-8").strip() for l in urlopen(labels_url).read().splitlines()]
assert len(labels) == 400, "Expected 400 Kinetics classes"

VIOLENT_KEYWORDS = [
    "punch", "kick", "wrestl", "slap", "fight", "boxing",
    "karate", "taekwondo", "judo", "kung fu", "mma", "mixed martial arts",
    "kickboxing", "capoeira"
]
EXCLUDE_PHRASES = ["punching bag", "arm wrestling"]

def is_violent_label(lbl: str):
    l = lbl.lower()
    if any(ex in l for ex in EXCLUDE_PHRASES): return False
    return any(k in l for k in VIOLENT_KEYWORDS)

VIOLENT_INDICES = [i for i, c in enumerate(labels) if is_violent_label(c)]
print(f"Violence-class count: {len(VIOLENT_INDICES)}")

def load_video_rgb(path, target=224):
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        h, w = f.shape[:2]
        # center square crop
        s = min(h, w)
        y0, x0 = (h - s)//2, (w - s)//2
        f = f[y0:y0+s, x0:x0+s]
        f = cv2.resize(f, (target, target))
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        frames.append(f)
    cap.release()
    return np.array(frames, dtype=np.uint8)

def make_clips(frames, clip_len=64, stride=32):
    T = len(frames)
    if T == 0: return []
    clips = []
    for start in range(0, max(1, T - clip_len + 1), stride):
        end = min(T, start + clip_len)
        clip = frames[start:end]
        if len(clip) < clip_len:
            pad = np.repeat(clip[-1][None, ...], clip_len - len(clip), axis=0)
            clip = np.concatenate([clip, pad], axis=0)
        clips.append((start, end, clip))
    return clips

def i3d_predict_rgb_clip(clip_uint8):
    # I3D expects float32 [0,1], shape [B,T,H,W,C]
    x = tf.cast(clip_uint8, tf.float32) / 255.0
    x = x[tf.newaxis, ...]
    logits = i3d(x)['default'][0]      # (400,)
    probs = tf.nn.softmax(logits).numpy()
    return probs

def score_video(path):
    frames = load_video_rgb(path, 224)
    clips = make_clips(frames, CLIP_LEN, STRIDE)
    violent_probs, top_preds = [], []

    for (_, _, clip) in clips:
        probs = i3d_predict_rgb_clip(clip)
        vprob = float(np.sum(probs[VIOLENT_INDICES]))
        violent_probs.append(vprob)
        topk = probs.argsort()[-3:][::-1]
        top_preds.append([(labels[i], float(probs[i])) for i in topk])

    violent_probs = np.array(violent_probs, dtype=np.float32)
    peak, mean = violent_probs.max() if len(violent_probs) else 0.0, violent_probs.mean() if len(violent_probs) else 0.0
    frac = float((violent_probs > 0.5).mean()) if len(violent_probs) else 0.0
    fight = (peak >= PEAK_THR) or ((mean >= MEAN_THR) and (frac >= FRAC_THR))
    return fight, dict(peak=peak, mean=mean, frac=frac, per_clip=violent_probs.tolist(), top=top_preds, nclips=len(violent_probs))

def annotate_and_save(src_path, verdict, stats, out_path):
    cap = cv2.VideoCapture(src_path)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    out = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
    banner = f"{'FIGHT' if verdict else 'NO FIGHT'} | peak:{stats['peak']:.2f} mean:{stats['mean']:.2f} frac:{stats['frac']:.2f}"
    color = (0, 200, 0) if verdict else (0, 0, 255)

    while True:
        ok, frame = cap.read()
        if not ok: break
        cv2.rectangle(frame, (8, 8), (w - 8, 80), (0, 0, 0), -1)
        cv2.putText(frame, banner, (20, 55), FONT, 1.2, color, 3, cv2.LINE_AA)
        out.write(frame)
    cap.release()
    out.release()

def run_one(path):
    verdict, stats = score_video(path)
    out_path = os.path.splitext(os.path.basename(path))[0] + "_i3d_annotated.mp4"
    annotate_and_save(path, verdict, stats, out_path)
    print(os.path.basename(path), "→",
          f"clips:{stats['nclips']}  peak:{stats['peak']:.3f}  mean:{stats['mean']:.3f}  frac>0.5:{stats['frac']:.2f}",
          "→", "FIGHT ✅" if verdict else "NO FIGHT ❌")
    if stats['nclips']:
        first = stats['top'][0]
        print("Top-3 (first clip):", ", ".join([f"{l}:{p:.2f}" for l,p in first]))
    display(Video(out_path, embed=True))

# ====== Upload your 2 videos ======
from google.colab import files
print("Upload fight.mp4 and nonfight.mp4 …")
_ = files.upload()

# ====== Run ======
run_one("fight.mp4")
run_one("nonfight.mp4")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.3/601.3 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorstore 0.1.76 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.4.1 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.17.1 wh

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [3]:
# --- Colab Setup ---
!pip install -q transformers accelerate av imageio

import torch
from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor
import imageio
import numpy as np
from google.colab import files

print("Upload fight.mp4 and nonfight.mp4 …")
# Check if files exist before prompting upload
import os
if not (os.path.exists("fight.mp4") and os.path.exists("nonfight.mp4")):
    _ = files.upload()
else:
    print("Files already exist. Skipping upload.")


# Load Model & Feature Extractor
model_name = "Nikeytas/Videomae-Crime-Detector-Ultra-V1"
model = VideoMAEForVideoClassification.from_pretrained(model_name)
feature_extractor = VideoMAEImageProcessor.from_pretrained(model_name)

# Helper: read video into frames and chunk into sequences of 16
def read_and_chunk_video(path, chunk_size=16):
    vid = imageio.get_reader(path, 'ffmpeg')
    frames = []
    chunks = []
    try:
        for i, frame in enumerate(vid):
            frames.append(frame)
            if len(frames) == chunk_size:
                chunks.append(frames)
                frames = [] # Reset for the next chunk
        # Add any remaining frames as the last chunk, padded if necessary
        if frames:
             while len(frames) < chunk_size:
                 # Pad with the last frame if the last chunk is smaller than chunk_size
                 if frames:
                     frames.append(frames[-1])
                 else:
                     # Should not happen if frames list is not empty
                     break
             chunks.append(frames)

    finally:
        vid.close()
    return chunks


# Predict function
def predict(path):
    video_chunks = read_and_chunk_video(path)
    if not video_chunks:
        print(f"Could not read or chunk any frames from {path}")
        return "No frames", [0.0, 0.0] # Return default values

    all_probs = []
    for chunk in video_chunks:
        # Process each chunk (list of 16 frames)
        inputs = feature_extractor(chunk, return_tensors="pt")

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)[0].tolist()
            all_probs.append(probs)

    # Average the probabilities across all chunks
    avg_probs = np.mean(all_probs, axis=0).tolist()

    # Determine the overall predicted label based on average probabilities
    # Assuming the model's output logits correspond to violence (index 1) vs non-violence (index 0)
    # Check if avg_probs has enough elements before accessing indices
    if len(avg_probs) > 1:
        predicted_label_index = int(np.argmax(avg_probs))
        if predicted_label_index < len(model.config.id2label):
            label = model.config.id2label[predicted_label_index]
        else:
             label = "Unknown"
             print(f"Warning: Predicted label index {predicted_label_index} out of bounds for id2label.")

    elif len(avg_probs) == 1:
         # Handle cases with a single output probability if applicable
         label = "Single Probability Output"
         print("Model output has only one probability.")
    else:
         label = "No Probabilities"


    print(f"{path} → {label} (avg probs across chunks: {avg_probs})")
    return label, avg_probs

# Run predictions
predict("fight.mp4")
predict("nonfight.mp4")

Upload fight.mp4 and nonfight.mp4 …
Files already exist. Skipping upload.
fight.mp4 → LABEL_0 (avg probs across chunks: [0.9989798290114249, 0.0010201698882625469])
nonfight.mp4 → LABEL_0 (avg probs across chunks: [0.9632797048737606, 0.036720304713526275])


('LABEL_0', [0.9632797048737606, 0.036720304713526275])